In [ ]:
# Install Cadabra if necessary, and enable Cadabra input in this notebook.
try:
    import cadabra2
except ImportError:
    %pip install -q cadabra2
%load_ext cadabra2_jupyter

# The Kaluza-Klein example from section 2.5 of hep-th/0701238

This example shows how to use [`split_index`](https://cadabra.science/manual/split_index.html) in a somewhat more complicated setting. 
We first declare the indices that we will use.

In [ ]:
{\mu,\nu,\rho,\sigma,\kappa,\lambda,\eta,\chi#}::Indices(full, position=independent);
{m,n,p,q,r,s,t,u,v,w,x,y,z,m#}::Indices(subspace, position=independent, parent=full);

Note the appearance of `parent=full`. This indicates that the indices in the second set
span a subspace of the indices in the first set. Also note that we have declared the indices
as `position=independent`, to prevent Cadabra from automatically raising or lowering them 
when canonicalising (as this does not really help us here).

The remaining declarations are standard,

In [ ]:
\partial{#}::PartialDerivative.
g_{\mu\nu}::Metric.
g^{\mu\nu}::InverseMetric.
g_{\mu? \nu?}::Symmetric.
g^{\mu? \nu?}::Symmetric.
h_{m n}::Metric.
h^{m n}::InverseMetric.
\delta^{\mu?}_{\nu?}::KroneckerDelta.
\delta_{\mu?}^{\nu?}::KroneckerDelta.
F_{m n}::AntiSymmetric.

We will want to expand the Riemann tensor in terms of the metric. The following two substitution
rules do the conversion from Riemann tensor to Christoffel symbol and from Christoffel symbol
to metric (a library with common substitution rules like these is in preparation).

In [ ]:
RtoG:= R^{\lambda?}_{\mu?\nu?\kappa?} -> 
 - \partial_{\kappa?}{ \Gamma^{\lambda?}_{\mu?\nu?} }
 + \partial_{\nu?}{ \Gamma^{\lambda?}_{\mu?\kappa?} }
 - \Gamma^{\eta}_{\mu?\nu?} \Gamma^{\lambda?}_{\kappa?\eta}
 + \Gamma^{\eta}_{\mu?\kappa?} \Gamma^{\lambda?}_{\nu?\eta};

Gtog:= \Gamma^{\lambda?}_{\mu?\nu?} ->
  (1/2) * g^{\lambda?\kappa} ( 
        \partial_{\nu?}{ g_{\kappa\mu?} } + \partial_{\mu?}{ g_{\kappa\nu?} } - \partial_{\kappa}{ g_{\mu?\nu?} } );

In this example we want to compute the Kaluza-Klein reduction of the $R_{m 4 n 4}$ component of the
Riemann tensor. So we enter this component and do the substitution that takes the Riemann tensor
to metrics. After each substitution, we distribute
products over sums. We also apply the product rule to distribute derivatives over factors in a
product.

In [ ]:
todo:= g_{m1 m} R^{m1}_{4 n 4} + g_{4 m} R^{4}_{4 n 4};

In [ ]:
substitute(_, RtoG)
substitute(_, Gtog)
distribute(_)
product_rule(_)
distribute(_)
sort_product(_);

We then split the greek indices into roman indices and the 4-th direction. Notice the use
of `repeat=True` which will keep doing this substitution until all greek indices
have been split. After that we again have to set all derivatives in the 4-th direction to zero.

In [ ]:
split_index(_, $\mu, m1, 4$, repeat=True)
substitute(_, $\partial_{4}{A??} -> 0$, repeat=True)
substitute(_, $\partial_{4 m?}{A??} -> 0$, repeat=True)
substitute(_, $\partial_{m? 4}{A??} -> 0$, repeat=True)
canonicalise(_);

Now comes the ansatz of the metric in terms of the Kaluza-Klein gauge field $A_\mu$ and the scalar $\phi$. 
We substitute this with the lines below. The output is not particularly enlightening...

In [ ]:
substitute(_, $g_{4 4} -> \phi$ )
substitute(_, $g_{m 4} -> \phi A_{m}$ )
substitute(_, $g_{4 m} -> \phi A_{m}$ )
substitute(_, $g_{m n} -> \phi**{-1} h_{m n} + \phi A_{m} A_{n}$ )
substitute(_, $g^{4 4} -> \phi**{-1} + \phi A_{m} h^{m n} A_{n}$ )
substitute(_, $g^{m 4} -> - \phi h^{m n} A_{n}$ )
substitute(_, $g^{4 m} -> - \phi h^{m n} A_{n}$ )
substitute(_, $g^{m n} -> \phi h^{m n}$ );

We now need to distribute derivatives over sums and use the product rule. This has to be done a few times,
so we wrap this in a `converge` block which applies the operations until the result no longer changes. The output 
of those operations is quite large, so we suppress it (by not writing semi-colons after the 
function calls or after the converge block).

In [ ]:
converge(todo):
   distribute(_)
   product_rule(_)
   canonicalise(_)

We then convert derivatives of the inverse metric to derivatives of the metric, and cleanup contractions
which lead to Kronecker deltas.

In [ ]:
substitute(_, $\partial_{p}{h^{n m}} h_{q m} -> - \partial_{p}{h_{q m}} h^{n m}$ )
collect_factors(_)
sort_product(_)
converge(todo):
	substitute(_, $h_{m1 m2} h^{m3 m2} -> \delta_{m1}^{m3}$, repeat=True )
	eliminate_kronecker(_)
	canonicalise(_)
;

This is almost the finaly result, but one would normally write it in terms of the field strength,
not the gauge potential. This can be done with one further substitution and some simplification, to 
get the result

In [ ]:
substitute(_, $\partial_{n}{A_{m}} -> 1/2*\partial_{n}{A_{m}} + 1/2*F_{n m} + 1/2*\partial_{m}{A_{n}}$ )
distribute(_)
sort_product(_)
canonicalise(_)
rename_dummies(_);